In [1]:
import pandas as pd
import json

# Load CSV
df = pd.read_csv(r"D:\Assignment\archive\baby_product.csv")

# Basic cleaning
df = df[df["review"].notna()]
df = df[df["review"].str.len() > 20]

# Group by product name (FULL DATASET)
grouped = df.groupby("name")

products = []

for i, (name, group) in enumerate(grouped):
    product = {
        "id": f"P{i+1}",
        "name": name,
        "features": ["baby-safe", "high quality"],  # you can improve later
        "reviews": group["review"].tolist(),
        "avg_rating": round(group["rating"].mean(), 2)
    }
    products.append(product)

# Save JSON
with open(r"D:\Assignment\archive\data_product.json", "w") as f:
    json.dump(products, f, indent=4)

print(f"✅ JSON created with {len(products)} products!")

✅ JSON created with 32253 products!


In [2]:
import json
import faiss
import numpy as np
import os
from datetime import datetime
from google import genai
from sentence_transformers import SentenceTransformer

# -----------------------------
# Config — paste your free key here
# Get it free at: https://aistudio.google.com/app/apikey
# -----------------------------
GEMINI_API_KEY = "Your API Key"
GEMINI_MODEL   = "gemini-2.5-flash"
client = genai.Client(api_key=GEMINI_API_KEY)

# -----------------------------
# Load Data
# -----------------------------
print("Loading data...\n")
with open(r"D:\Assignment\archive\data_product.json") as f:
    products = json.load(f)

names = [p["name"] for p in products]
print("Available Products:\n")
for i, name in enumerate(names):
    print(f"  {i+1}. {name}")

# -----------------------------
# User Input
# -----------------------------
print()
p1 = input("Enter Product 1 name: ").strip()
p2 = input("Enter Product 2 name: ").strip()

if p1 == p2:
    print("\n⚠️  Please choose two different products.")
    exit(1)

# -----------------------------
# Find products & compute avg ratings
# -----------------------------
def get_avg_rating(product_name, all_products):
    """Find a product by name and return its average rating."""
    for p in all_products:
        if p["name"].lower() == product_name.lower():
            rating = p.get("avg_rating", None)
            if isinstance(rating, list):
                numeric = []
                for r in rating:
                    try:
                        numeric.append(float(r))
                    except (ValueError, TypeError):
                        pass
                return round(sum(numeric) / len(numeric), 2) if numeric else "N/A"
            elif isinstance(rating, (int, float)):
                return rating
            elif isinstance(rating, str):
                try:
                    return float(rating)
                except ValueError:
                    return "N/A"
            return "N/A"
    return "Not found"

avg_rating_p1 = get_avg_rating(p1, products)
avg_rating_p2 = get_avg_rating(p2, products)

# -----------------------------
# Load Embedding Model
# -----------------------------
print("\nLoading embedding model...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# -----------------------------
# Prepare texts for FAISS
# -----------------------------
texts    = []
metadata = []
for p in products:
    review_text = " ".join(p.get("reviews", [])[:3])
    clean_text  = f"Product: {p['name']}. Reviews: {review_text}"
    texts.append(clean_text)
    metadata.append(p)

# -----------------------------
# Build FAISS Index
# -----------------------------
print("Building search index...")
embeddings = embed_model.encode(texts)
embeddings = np.array(embeddings, dtype="float32")
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

# -----------------------------
# Retrieval Function
# -----------------------------
def retrieve(query, k=4):
    q_emb = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, k)
    return [metadata[i] for i in indices[0]]

# -----------------------------
# Build Context
# -----------------------------
query         = f"{p1} vs {p2} comparison features reviews"
context_items = retrieve(query)
context_text  = ""
for item in context_items:
    reviews    = "\n  - ".join(item.get("reviews", [])[:3])
    price      = item.get("price", "N/A")
    avg_rating = item.get("avg_rating", "N/A")
    context_text += (
        f"Product: {item['name']}\n"
        f"Price: {price}\n"
        f"Avg Rating: {avg_rating}\n"
        f"Reviews:\n  - {reviews}\n\n"
    )

# -----------------------------
# Generate with Gemini
# -----------------------------
print(f"\nComparing '{p1}' vs '{p2}' using Gemini...\n")
prompt = f"""You are a helpful product comparison assistant.
Compare these two products:
- Product A: {p1}
- Product B: {p2}

Here is relevant product data:
{context_text}

Give a structured comparison:
1. Pros of {p1}
2. Pros of {p2}
3. Cons of {p1}
4. Cons of {p2}
5. Best pick and why

Be concise and base your answer only on the data above."""

try:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    result = response.text
except Exception as e:
    print(f"❌ Gemini API error: {e}")
    print("   Check your API key at https://aistudio.google.com/app/apikey")
    exit(1)

# -----------------------------
# Build Output
# -----------------------------
output_lines = []
output_lines.append("=" * 52)
output_lines.append("  COMPARISON RESULT")
output_lines.append("=" * 52)
output_lines.append(f"\n⭐ Average Ratings:")
output_lines.append(f"   {p1:<25} →  {avg_rating_p1} / 5")
output_lines.append(f"   {p2:<25} →  {avg_rating_p2} / 5")
output_lines.append("")
output_lines.append(result)
output_lines.append("=" * 52)

final_output = "\n".join(output_lines)

# -----------------------------
# Print to Console
# -----------------------------
print(final_output)

# -----------------------------
# Save to File — D:\Assignment\archive
# -----------------------------
SAVE_DIR = r"D:\Assignment\archive"

safe_p1   = "".join(c if c.isalnum() or c in " _-" else "_" for c in p1)[:30]
safe_p2   = "".join(c if c.isalnum() or c in " _-" else "_" for c in p2)[:30]
safe_p1   = safe_p1.replace(" ", "_")
safe_p2   = safe_p2.replace(" ", "_")
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename  = f"comparison_{safe_p1}_vs_{safe_p2}_{timestamp}.txt"
save_path = os.path.join(SAVE_DIR, filename)

with open(save_path, "w", encoding="utf-8") as f:
    f.write(final_output)

print(f"\n✅ Result saved to: {save_path}")

Loading data...

Available Products:

  1. # 1 Digital Baby Thermometer - With FREE LIFETIME GUARANTEE: Using Non-Contact Infra Red (IR) Technology giving you instant and accurate temperature readings - Protect your BABY from FEVER and ILLNESS by using the DUAL Colour Digital Display and AUDIO ALARM function. Instant temperature reading from Baby Forehead within 1cm to 6cm distance - AUDIO Warning will trigger when temperature exceeds (37.5&#8451;) - Built in Memory to store 20 Readings - Fully FDA Approved &amp; (C.E.) Rated for Safety - Sold in one neat compact size 0.23kg weight. Durable Silicone touch buttons for durability - The non-contact design allows an instant temperature reading while your baby is sleeping. Can also be used to take the temperature of Baby Milk Bottles - In STOCK &amp; SHIPS today for FREE with Amazon.com
  2. #1 Adjustable Back Seat Baby Safety Mirror - Easy To Fit - Mirror Attaches In Seconds To Rear Seat Head Rest And Rotates And Pivots For A Clear View - 

Enter Product 1 name:  timi &amp; leslie Tag - a - Long Diaper Bag, Sahara Brown
Enter Product 2 name:  timi &amp; leslie Rachel 7-Piece Diaper Bag Set, Black



Loading embedding model...
Building search index...

Comparing 'timi &amp; leslie Tag - a - Long Diaper Bag, Sahara Brown' vs 'timi &amp; leslie Rachel 7-Piece Diaper Bag Set, Black' using Gemini...

  COMPARISON RESULT

⭐ Average Ratings:
   timi &amp; leslie Tag - a - Long Diaper Bag, Sahara Brown →  4.15 / 5
   timi &amp; leslie Rachel 7-Piece Diaper Bag Set, Black →  4.92 / 5

Here's a comparison of the two products based on the provided data:

**Please note:** The product data provided for "timi & leslie Rachel 7-Piece Diaper Bag Set, Black" is actually for "timi & leslie Jessica 7-Piece Diaper Bag Set, Black". The comparison will use the data for the Jessica bag as provided.

---

### 1. Pros of timi & leslie Tag - a - Long Diaper Bag, Sahara Brown
*   Colors are brighter and bolder than pictured (positive for one reviewer).
*   Includes stroller straps and changing pad.
*   Fabric is super durable and easy to clean.
*   Features many pockets and is easy to use, overcoming issue